In [1]:
import os, re, json, random, string, hashlib, urllib.parse, time, warnings, glob
import numpy as np, pandas as pd, requests
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

RS = 42
random.seed(RS); np.random.seed(RS)
for d in ['../data','../figures','../results','../corpus']:
    os.makedirs(d, exist_ok=True)
print("Setup complete.")

Setup complete.


In [2]:
PATT = '../data/patt'
FAMILIES = {'sqli':['SQL Injection'], 'xss':['XSS Injection'], 'cmdi':['Command Injection']}

def clean(line):
    l = line.strip()
    if not l or len(l) < 4 or len(l) > 400: return None
    if l.startswith(('#','```','|','<!--','---','===','* ','- [','> ')): return None
    if l.startswith(('http://','https://')): return None
    if re.match(r'^[A-Z][a-z]+ ', l) and not any(c in l for c in "'\"<>();=|&"): return None
    return l

def load_family(dirs, limit=4000):
    out = []
    for d in dirs:
        base = os.path.join(PATT, d)
        if not os.path.isdir(base): continue
        for f in glob.glob(base+'/**/*.txt', recursive=True):
            try:
                for line in open(f, encoding='utf-8', errors='ignore'):
                    c = clean(line)
                    if c: out.append(c)
            except Exception: pass
    seen, uniq = set(), []
    for p in out:
        if p not in seen: seen.add(p); uniq.append(p)
    return uniq[:limit]

payloads = {f: load_family(d) for f, d in FAMILIES.items()}
for f, v in payloads.items(): print(f"{f:6s}: {len(v):>5,}")
json.dump(payloads, open('../corpus/payloads.json','w'))

sqli  : 1,039
xss   : 2,101
cmdi  :   402


In [3]:
FIRST=['james','maria','ana','carlos','sophie','david','elena','luis','priya','wei','omar',
       'sara','nikhil','anna','marco','yuki','hassan','lena','pablo','ravi','tomas','ines']
LAST=['smith','garcia','muller','rossi','chen','patel','kumar','silva','novak','dubois',
      'ahmed','kowalski','nielsen','tanaka','okafor','ivanov','moreau','costa']
CITIES=['madrid','london','berlin','paris','rome','mumbai','tokyo','lisbon','vienna','oslo']
TERMS=['apple','juice','banana','gift','offer','discount','organic','fresh','summer',
       'premium','seasonal','gluten free','vegan','sugar free','cold pressed','family pack']
PRODUCTS=['vino rioja','jamon iberico','apple juice','green smoothie','olive oil',
          'manchego cheese','dark chocolate','flower honey','bomba rice','sweet paprika']

def uid(n=6):  return ''.join(random.choices(string.digits, k=n))
def pw():      return ''.join(random.choices(string.ascii_letters+string.digits, k=random.randint(10,20)))
def person():  return f"{random.choice(FIRST)}.{random.choice(LAST)}{random.randint(1,999)}"
def sentence(k=None):
    k = k or random.randint(4,14)
    return '+'.join(random.choice(TERMS+PRODUCTS+FIRST+CITIES).replace(' ','+') for _ in range(k))

APPS = {
 'dvwa': {'base':'http://localhost:8080', 'benign':[
   ('/vulnerabilities/sqli/',  lambda: f"id={random.randint(1,5000)}&Submit=Submit&session={uid(8)}&ref={sentence(2)}"),
   ('/vulnerabilities/xss_r/', lambda: f"name={sentence(random.randint(2,8))}&token={uid(10)}"),
   ('/login.php',              lambda: f"username={person()}&password={pw()}&Login=Login&redirect=%2Fvulnerabilities%2Fsqli%2F&csrf={uid(12)}"),
   ('/vulnerabilities/exec/',  lambda: f"ip={random.randint(1,254)}.{random.randint(0,254)}.{random.randint(0,254)}.{random.randint(1,254)}&Submit=Submit&comment={sentence(random.randint(3,10))}"),
   ('/guestbook.php',          lambda: f"txtName={person()}&mtxMessage={sentence(random.randint(6,18))}&btnSign=Sign+Guestbook"),
   ('/vulnerabilities/upload/',lambda: f"file=report_{uid(5)}.pdf&MAX_FILE_SIZE={random.choice([1000,5000,100000])}&description={sentence(random.randint(4,12))}")],
  'inject':[('/vulnerabilities/sqli/','id','&Submit=Submit'),
            ('/vulnerabilities/xss_r/','name',''),
            ('/vulnerabilities/exec/','ip','&Submit=Submit')]},

 'juice': {'base':'http://localhost:3000', 'benign':[
   ('/rest/products/search', lambda: f"q={sentence(random.randint(2,7))}&page={random.randint(1,60)}&sort={random.choice(['price','name','rating'])}"),
   ('/api/Products',         lambda: f"id={random.randint(1,900)}&limit={random.choice([10,20,50])}&fields=name,price,description&filter={sentence(2)}"),
   ('/rest/user/login',      lambda: f"email={person()}%40mail.com&password={pw()}&remember={random.choice(['true','false'])}"),
   ('/api/Feedbacks',        lambda: f"rating={random.randint(1,5)}&comment={sentence(random.randint(6,20))}&userId={uid(4)}"),
   ('/rest/basket',          lambda: f"productId={random.randint(1,500)}&quantity={random.randint(1,20)}&basketId={uid(4)}&coupon={uid(6)}&note={sentence(random.randint(3,9))}"),
   ('/api/Addresss',         lambda: f"city={random.choice(CITIES)}&zip={uid(5)}&street=oak+{random.randint(1,300)}&country={random.choice(['ES','UK','DE','FR','IN'])}&fullName={person()}")],
  'inject':[('/rest/products/search','q',''),
            ('/api/Feedbacks','comment','')]},

 'bwapp': {'base':'http://localhost:8081', 'benign':[
   ('/sqli_1.php',   lambda: f"title={sentence(random.randint(2,8))}&action=search&limit={random.randint(5,50)}"),
   ('/xss_get.php',  lambda: f"firstname={random.choice(FIRST)}{random.randint(1,999)}&lastname={random.choice(LAST)}&form=submit&message={sentence(random.randint(4,14))}"),
   ('/portal.php',   lambda: f"city={random.choice(CITIES)}&ref={uid(6)}&lang={random.choice(['en','es','de','fr'])}&theme={random.choice(['light','dark'])}"),
   ('/commandi.php', lambda: f"target=www.example{random.randint(1,999)}.com&form=submit&opts={sentence(3)}"),
   ('/search.php',   lambda: f"keyword={sentence(random.randint(2,9))}&cat={random.randint(1,40)}&page={random.randint(1,25)}&desc={random.choice(['asc','desc'])}"),
   ('/user.php',     lambda: f"login={person()}&token={uid(8)}&bio={sentence(random.randint(5,16))}")],
  'inject':[('/sqli_1.php','title','&action=search'),
            ('/xss_get.php','firstname','&form=submit'),
            ('/commandi.php','target','&form=submit')]},
}
print(f"{len(APPS)} applications configured.")

3 applications configured.


In [4]:
SESSIONS = {}

# DVWA
s = requests.Session()
r = s.get('http://localhost:8080/login.php', timeout=10)
tok = re.search(r"user_token'\s*value='([^']+)'", r.text)
data = {'username':'admin','password':'password','Login':'Login'}
if tok: data['user_token'] = tok.group(1)
s.post('http://localhost:8080/login.php', data=data, timeout=10)
s.get('http://localhost:8080/security.php', timeout=10)
s.post('http://localhost:8080/security.php',
       data={'security':'low','seclev_submit':'Submit'}, timeout=10)
SESSIONS['dvwa'] = s

# bWAPP
s2 = requests.Session()
s2.get('http://localhost:8081/login.php', timeout=10)
s2.post('http://localhost:8081/login.php',
        data={'login':'bee','password':'bug','security_level':'0','form':'submit'}, timeout=10)
SESSIONS['bwapp'] = s2

# Juice Shop needs no login for these endpoints
SESSIONS['juice'] = requests.Session()

print("Verifying all injection points return 200:")
allok = True
for app, cfg in APPS.items():
    print(f"\n{app}")
    for path, _, _ in cfg['inject']:
        try:
            r = SESSIONS[app].get(cfg['base']+path, timeout=5, allow_redirects=False)
            ok = r.status_code == 200
            allok &= ok
            print(f"  {path:32s} -> {r.status_code}  {'OK' if ok else 'PROBLEM'}")
        except Exception as e:
            allok = False
            print(f"  {path:32s} -> ERROR ({type(e).__name__})")
print("\nAll endpoints ready." if allok else "\nFix the failures before capturing.")

Verifying all injection points return 200:

dvwa
  /vulnerabilities/sqli/           -> 200  OK
  /vulnerabilities/xss_r/          -> 200  OK
  /vulnerabilities/exec/           -> 200  OK

juice
  /rest/products/search            -> 200  OK
  /api/Feedbacks                   -> 200  OK

bwapp
  /sqli_1.php                      -> 200  OK
  /xss_get.php                     -> 200  OK
  /commandi.php                    -> 200  OK

All endpoints ready.


In [5]:
def build(app, n, attack):
    cfg = APPS[app]; out = []
    for _ in range(n):
        if attack:
            fam = random.choice(list(payloads)); p = random.choice(payloads[fam])
            path, param, extra = random.choice(cfg['inject'])
            other = f"&session={uid(8)}" if random.random() < .5 else ""
            note  = f"&note={sentence(random.randint(2,6))}" if random.random() < .4 else ""
            q = f"{param}={urllib.parse.quote_plus(p)}{extra}{other}{note}"
            out.append({'app':app,'method':'GET','path':path,'query':q,'body':'',
                        'label':1,'attack_type':fam,'source':'patt'})
        else:
            path, fn = random.choice(cfg['benign'])
            out.append({'app':app,'method':'GET','path':path,'query':fn(),'body':'',
                        'label':0,'attack_type':'benign','source':'scripted_browse'})
    return out

def send(records, drop_5xx, pause=0.01):
    kept = []; redirected = 0
    for i, r in enumerate(records):
        sess = SESSIONS[r['app']]
        url = APPS[r['app']]['base'] + r['path'] + '?' + r['query']
        try:
            resp = sess.get(url, timeout=4, allow_redirects=False)
            r['status'] = resp.status_code
            if resp.status_code == 302:          # bounced to login, not real traffic
                redirected += 1; continue
            if drop_5xx and resp.status_code >= 500: continue
            kept.append(r)
        except Exception: continue
        if pause: time.sleep(pause)
        if (i+1) % 2000 == 0: print(f"    {i+1:,}/{len(records):,}")
    if redirected: print(f"    WARNING: {redirected:,} requests were redirected (session lost?)")
    return kept

records = []
for app in APPS:
    print(f"\n{app} — benign")
    records += send(build(app, 14000, False), drop_5xx=True)
    print(f"{app} — attacks")
    records += send(build(app, 4500, True), drop_5xx=False)

json.dump(records, open('../corpus/raw_capture.json','w'))
print(f"\nCaptured {len(records):,} requests")


dvwa — benign
    2,000/14,000
    4,000/14,000
    6,000/14,000
    8,000/14,000
    10,000/14,000
    12,000/14,000
    14,000/14,000
dvwa — attacks
    2,000/4,500
    4,000/4,500

juice — benign
    6,000/14,000
    8,000/14,000
    10,000/14,000
    12,000/14,000
    14,000/14,000
juice — attacks
    2,000/4,500
    4,000/4,500

bwapp — benign
    2,000/14,000
    4,000/14,000
    6,000/14,000
    8,000/14,000
    10,000/14,000
    12,000/14,000
    14,000/14,000
bwapp — attacks
    2,000/4,500
    4,000/4,500

Captured 53,113 requests


In [6]:
def normalise(path, query, body=''):
    return urllib.parse.unquote_plus(f"{path} {query} {body}".strip())

recs = json.load(open('../corpus/raw_capture.json'))
df = pd.DataFrame(recs)
df['payload'] = [normalise(r['path'], r['query'], r.get('body','')) for r in recs]
df = df[df.payload.str.len() >= 3]
df['fp'] = df.payload.map(lambda s: hashlib.md5(s.encode('utf-8','ignore')).hexdigest())
before = len(df); df = df.drop_duplicates(subset='fp')
print(f"dedup: {before:,} -> {len(df):,}")

df = df[['app','method','path','query','payload','label','attack_type','source']]
df.to_csv('../data/webinject2026.csv', index=False)

print(f"\nTotal {len(df):,} | benign {(df.label==0).sum():,} | attack {(df.label==1).sum():,}")
print("\nBy app:\n", df.app.value_counts().to_string())
print("\nBy attack type:\n", df.attack_type.value_counts().to_string())

ratio = (df.label==1).mean()
bl = df[df.label==0].payload.str.len().median()
al = df[df.label==1].payload.str.len().median()
gap = abs(bl-al)/max(bl,al)
print(f"\nCHECKS")
print(f"  attack ratio {ratio:.3f}   {'PASS' if .15<=ratio<=.6 else 'FAIL'}")
print(f"  applications {df.app.nunique()}       {'PASS' if df.app.nunique()>=3 else 'WARN'}")
print(f"  duplicates {df.payload.duplicated().sum()}       {'PASS' if not df.payload.duplicated().any() else 'FAIL'}")
print(f"  length gap {gap:.3f}    {'PASS' if gap<=0.15 else 'FAIL'}")
print(f"  size {len(df):,}      {'PASS' if len(df)>=30000 else 'WARN — aim for 30k+'}")

dedup: 53,113 -> 52,696

Total 52,696 | benign 39,648 | attack 13,048

By app:
 app
bwapp    18373
dvwa     18361
juice    15962

By attack type:
 attack_type
benign    39648
xss        4513
sqli       4335
cmdi       4200

CHECKS
  attack ratio 0.248   PASS
  applications 3       PASS
  duplicates 0       PASS
  length gap 0.000    PASS
  size 52,696      PASS


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

L = df.payload.str.len().values.reshape(-1,1)
a,b,c,d_ = train_test_split(L, df.label.values, test_size=.3, stratify=df.label, random_state=RS)
m = XGBClassifier(n_estimators=50, random_state=RS, eval_metric='logloss', verbosity=0).fit(a,c)
f_len = f1_score(d_, m.predict(b))*100
print(f"F1 using ONLY payload length: {f_len:.2f}%")
print("PASS — length carries little signal" if f_len < 60 else
      "FAIL — corpus broken. Lengthen benign requests in Cell 3 and re-capture.")

F1 using ONLY payload length: 24.18%
PASS — length carries little signal
